# Crash Hotspot Detection in Dubai using DBSCAN and Severity Weighting

Implementation of the pipeline described in:

> M. Alblooshi, N. Nour, and K. Elgazzar, "Crash Hotspot Detection in Dubai using DBSCAN and Severity Weighting," *2026 IEEE International Conference on Smart Mobility (SM)*, Al Alamein City, Egypt, 2026.

**Pipeline:** Traffic Incident Dataset → Preprocessing → DBSCAN Spatial Clustering (Haversine) → Severity-Weighted Risk Scoring → Temporal Analysis (peak time window) → Spatial Annotation (reverse geocoding) & Cluster Visualization.

> ⚠️ **Data note:** This notebook expects the Dubai Pulse / Dubai Police "Traffic Incidents" open dataset (see [dubaipulse.gov.ae](https://www.dubaipulse.gov.ae/data/dp-traffic/dp_traffic_incidents-open)). Column names below are based on the dataset's public schema description in the paper — adjust `COLUMN MAPPING` in the cell below to match the exact CSV you download, since open-data portals occasionally rename fields.


In [ ]:
# =========================
# 🚀 1. Setup
# =========================
import pandas as pd
import numpy as np
import re
from datetime import datetime

from sklearn.cluster import DBSCAN

# Optional (install if not present): pip install geopy folium --break-system-packages
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import folium

np.random.seed(42)


## 📂 2. Load Dataset

Update `DATA_PATH` to point to your downloaded CSV, and adjust `COLUMN MAPPING`
if your file's column headers differ from the ones below.


In [ ]:
# =========================
# 📂 Load Dataset
# =========================
DATA_PATH = "traffic_incidents.csv"  # <-- update to your local file

# Column mapping: rename your dataset's columns to these standard names
COLUMN_MAPPING = {
    "accident_id": "incident_id",
    "description": "description_ar",   # incident description (Arabic text in the source dataset)
    "latitude": "latitude",
    "longitude": "longitude",
    "acci_time": "incident_datetime",   # timestamp of the accident
    "load_time": "load_datetime",       # timestamp the record was loaded
}

df = pd.read_csv(DATA_PATH)
df = df.rename(columns={v: k for k, v in COLUMN_MAPPING.items() if v in df.columns})

print(f"Loaded {len(df)} raw records")
df.head()


## 🧹 3. Preprocessing

- Drop records with missing/invalid coordinates
- Filter to Dubai's approximate geographic boundary
- Parse timestamps to datetime


In [ ]:
# =========================
# 🧹 Preprocessing
# =========================
# Dubai's approximate bounding box (lat, lon)
DUBAI_LAT_RANGE = (24.70, 25.40)
DUBAI_LON_RANGE = (54.90, 55.65)

df = df.dropna(subset=["latitude", "longitude"]).copy()
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
df = df.dropna(subset=["latitude", "longitude"])

df = df[
    df["latitude"].between(*DUBAI_LAT_RANGE) &
    df["longitude"].between(*DUBAI_LON_RANGE)
]

df["incident_datetime"] = pd.to_datetime(df["incident_datetime"], errors="coerce")
df = df.dropna(subset=["incident_datetime"])

print(f"{len(df)} records remain after cleaning and boundary filtering")


## ⚖️ 4. Severity Weighting

Incidents are assigned a severity weight via keyword matching on the description text,
per the paper: **minor = 1, moderate = 2, severe = 3**, with a default weight of 1
for uncategorized incidents. Keyword lists below are a starting point — refine them
to match the actual vocabulary in your dataset's `description_ar` field (Arabic text
in the source Dubai Pulse dataset; English keywords included for readability/adaptation).


In [ ]:
# =========================
# ⚖️ Severity Weighting
# =========================
# NOTE: adjust these keyword lists to match your dataset's actual description text.
SEVERE_KEYWORDS = ["fatal", "death", "severe", "وفاة", "خطير", "جسيم"]
MODERATE_KEYWORDS = ["injury", "injured", "moderate", "إصابة", "متوسط"]
MINOR_KEYWORDS = ["minor", "material", "damage", "بسيط", "مادي"]

def severity_weight(description: str) -> int:
    if not isinstance(description, str):
        return 1  # default weight
    text = description.lower()
    if any(kw in text for kw in SEVERE_KEYWORDS):
        return 3
    if any(kw in text for kw in MODERATE_KEYWORDS):
        return 2
    if any(kw in text for kw in MINOR_KEYWORDS):
        return 1
    return 1  # default weight for unmatched categories

df["severity_weight"] = df["description"].apply(severity_weight) if "description" in df.columns else 1

df["severity_weight"].value_counts().sort_index()


## 📍 5. Spatial Clustering — DBSCAN with Haversine Distance

DBSCAN groups incidents by spatial density rather than shape, and does not require
specifying the number of clusters in advance. The Haversine metric accounts for
Earth's curvature, so `eps` is expressed as a real-world radius in meters.


In [ ]:
# =========================
# 📍 DBSCAN Spatial Clustering
# =========================
EPS_METERS = 150      # neighborhood radius in meters — tune per dataset density
MIN_PTS = 5            # minimum points to form a dense cluster core
EARTH_RADIUS_M = 6371000

coords = np.radians(df[["latitude", "longitude"]].values)
eps_rad = EPS_METERS / EARTH_RADIUS_M

db = DBSCAN(eps=eps_rad, min_samples=MIN_PTS, metric="haversine")
df["cluster"] = db.fit_predict(coords)

n_clusters = len(set(df["cluster"])) - (1 if -1 in df["cluster"].values else 0)
n_noise = (df["cluster"] == -1).sum()

print(f"Found {n_clusters} clusters")
print(f"{n_noise} points classified as noise/outliers")


## 🚨 6. Severity-Weighted Risk Scoring

Each cluster's risk score is the sum of its incidents' severity weights:

$$\text{Risk}(c) = \sum_{i \in c} w_i$$

This means a small cluster of severe crashes can outrank a larger cluster of minor ones.


In [ ]:
# =========================
# 🚨 Risk Scoring
# =========================
clustered = df[df["cluster"] != -1].copy()

cluster_summary = clustered.groupby("cluster").agg(
    incident_count=("cluster", "size"),
    risk_score=("severity_weight", "sum"),
    centroid_lat=("latitude", "mean"),
    centroid_lon=("longitude", "mean"),
).reset_index()

cluster_summary = cluster_summary.sort_values("risk_score", ascending=False).reset_index(drop=True)

# Risk band for visualization (High / Medium / Low), by tercile of risk score
cluster_summary["risk_band"] = pd.qcut(
    cluster_summary["risk_score"], q=3, labels=["Low", "Medium", "High"]
)

cluster_summary.head(10)


## 🕒 7. Temporal Analysis — Peak Risk Window

Incident timestamps are binned into 2-hour windows; the most common window per
cluster is taken as that hotspot's peak-risk period, per the paper's methodology.


In [ ]:
# =========================
# 🕒 Temporal Analysis
# =========================
def two_hour_window(ts: pd.Timestamp) -> str:
    start_hour = (ts.hour // 2) * 2
    end_hour = start_hour + 2
    return f"{start_hour:02d}:00–{end_hour:02d}:00"

clustered["time_window"] = clustered["incident_datetime"].apply(two_hour_window)

peak_windows = (
    clustered.groupby("cluster")["time_window"]
    .agg(lambda x: x.value_counts().idxmax())
    .rename("peak_time_window")
)

cluster_summary = cluster_summary.merge(peak_windows, on="cluster", how="left")
cluster_summary.head(10)


## 🗺️ 8. Reverse Geocoding — Human-Readable Hotspot Labels

Each cluster's centroid is reverse-geocoded via the Nominatim API to produce a
descriptive label (nearby road, neighborhood, or district), matching the paper's approach.

> ⏱️ Nominatim's public API is rate-limited — this step can take a few minutes for
> many clusters. A `RateLimiter` is used to stay within the 1 request/second policy.


In [ ]:
# =========================
# 🗺️ Reverse Geocoding
# =========================
geolocator = Nominatim(user_agent="dubai-crash-hotspot-detection")
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)

def label_cluster(row) -> str:
    try:
        location = reverse((row["centroid_lat"], row["centroid_lon"]), language="en")
        if location and location.raw.get("address"):
            addr = location.raw["address"]
            parts = [addr.get("road"), addr.get("suburb") or addr.get("neighbourhood"), addr.get("city")]
            return ", ".join([p for p in parts if p])
        return "Unknown location"
    except Exception:
        return "Geocoding failed"

cluster_summary["location_label"] = cluster_summary.apply(label_cluster, axis=1)
cluster_summary.head(10)


## 📈 9. Cluster Visualization

Plots hotspot centroids on an interactive map, colored by risk band
(🔴 High / 🟠 Medium / 🟢 Low) and sized by incident count — matching Fig. 2 in the paper.


In [ ]:
# =========================
# 📈 Interactive Map
# =========================
RISK_COLORS = {"High": "red", "Medium": "orange", "Low": "green"}

dubai_center = [df["latitude"].mean(), df["longitude"].mean()]
m = folium.Map(location=dubai_center, zoom_start=11, tiles="cartodbpositron")

for _, row in cluster_summary.iterrows():
    folium.CircleMarker(
        location=[row["centroid_lat"], row["centroid_lon"]],
        radius=4 + row["incident_count"] ** 0.5,
        color=RISK_COLORS[row["risk_band"]],
        fill=True,
        fill_opacity=0.7,
        popup=(
            f"<b>{row.get('location_label', 'Hotspot')}</b><br>"
            f"Incidents: {row['incident_count']}<br>"
            f"Risk score: {row['risk_score']}<br>"
            f"Peak window: {row.get('peak_time_window', 'N/A')}"
        ),
    ).add_to(m)

m


## 💾 10. Export Results


In [ ]:
# =========================
# 💾 Export
# =========================
cluster_summary.to_csv("crash_hotspots_summary.csv", index=False)
m.save("crash_hotspots_map.html")

print("Saved: crash_hotspots_summary.csv, crash_hotspots_map.html")
print(f"\nTop 5 highest-risk hotspots:")
cluster_summary[["location_label", "risk_score", "incident_count", "peak_time_window", "risk_band"]].head(5)
